# MNIST Outlier Detection with RankOD

This notebook demonstrates **RankOD's ability to detect out-of-distribution samples** using the MNIST handwritten digits dataset.

## Scenario

We will:
1. Load the MNIST dataset (digits 0-9)
2. **Hold out one class** (digit "9") to simulate outliers
3. **Train RankOD only on the remaining classes** (digits 0-8)
4. Score all samples (including the held-out class)
5. Verify that the held-out class receives the highest outlier scores

This demonstrates that RankOD can effectively identify data that doesn't belong to the training distribution - a critical capability for real-world anomaly detection where you don't have labeled anomalies.

## Setup and Configuration

First, we'll import the necessary libraries and set up our experiment configuration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from hirank import RankOD

# Set random seed for reproducibility
np.random.seed(42)

# Configuration
HELD_OUT_CLASS = 9  # Which digit to hold out as "outliers"
N_NEIGHBORS = 15
MAX_RANK = 100
PCA_COMPONENTS = 250  # Reduce 784 dimensions to 250
CONTAMINATION = 0.1  # Expected proportion of outliers

print(f"✓ Libraries imported successfully")
print(f"Configuration:")
print(f"  Held-out class: {HELD_OUT_CLASS}")
print(f"  n_neighbors: {N_NEIGHBORS}, max_rank: {MAX_RANK}")
print(f"  PCA components: {PCA_COMPONENTS}")

## Load MNIST Dataset

We'll load 10,000 samples from MNIST for faster computation. The dataset contains 28×28 grayscale images (784 features) of handwritten digits 0-9.

In [ ]:
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X = mnist.data.to_numpy()
y = mnist.target.astype(int).to_numpy()

# Take a subset for faster computation
n_samples = 10000
indices = np.random.choice(len(X), n_samples, replace=False)
X, y = X[indices], y[indices]

# Normalize to [0, 1]
X = X / 255.0

print(f"✓ Loaded {len(X):,} samples with {X.shape[1]} features")
print(f"Class distribution: {dict(zip(range(10), np.bincount(y)))}")

## Split Data: Normal vs Outliers

Now we split the data into two sets:
- **Normal (training)**: All digits except 9 → these are "normal" samples  
- **Outliers (held-out)**: Only digit 9 → these will be our "outliers"

The detector will **never see digit 9 during training**, so it should score it as anomalous.

In [ ]:
# Split into normal and outlier sets
normal_mask = y != HELD_OUT_CLASS
outlier_mask = y == HELD_OUT_CLASS

X_normal = X[normal_mask]
y_normal = y[normal_mask]
X_outlier = X[outlier_mask]
y_outlier = y[outlier_mask]

print(f"✓ Data split complete")
print(f"Normal (training): {len(X_normal):,} samples (classes {sorted(set(y_normal))})")
print(f"Outliers (held-out class {HELD_OUT_CLASS}): {len(X_outlier):,} samples")
print(f"\nOutlier proportion: {len(X_outlier)/len(X):.1%}")

## Apply PCA Dimensionality Reduction

To speed up computation, we reduce dimensionality from 784 to 50 using PCA. This retains ~80% of the variance while making the algorithm much faster.

**Important**: We fit PCA only on the normal data (digits 0-8), not on the outliers!

In [ ]:
print(f"Applying PCA: {X_normal.shape[1]} → {PCA_COMPONENTS} dimensions...")

# Fit PCA on normal data only (don't use outliers!)
pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_normal)
X_all_pca = pca.transform(X)

variance_explained = pca.explained_variance_ratio_.sum()
print(f"✓ PCA complete")
print(f"Variance explained: {variance_explained:.1%}")
print(f"Training data shape: {X_train_pca.shape}")

## Train RankOD on Normal Data Only

We train the detector **only on digits 0-8**, never showing it digit 9. This simulates a real scenario where you don't have examples of anomalies during training.

We use `precompute_neighbors=True` for speed optimization (7-29x faster for scoring new data).

In [ ]:
print("Training RankOD detector...")
detector = RankOD(
    n_neighbors=N_NEIGHBORS,
    max_rank=MAX_RANK,
    contamination=CONTAMINATION,
    precompute_neighbors=True,  # Use fast mode for speed
    random_state=42,
    verbose=False
)

detector.fit(X_train_pca)
print("✓ Training complete!")

## Score All Samples

Now we score **all samples**, including the held-out digit 9 that the detector has never seen. Higher scores indicate more likely outliers.

In [ ]:
print(f"Scoring all {len(X_all_pca):,} samples...")
scores = detector.score_samples(X_all_pca)

# Split scores into normal vs outlier
normal_scores = scores[normal_mask]
outlier_scores = scores[outlier_mask]

print("✓ Scoring complete!")
print(f"Score range: [{scores.min():.2f}, {scores.max():.2f}]")

## Evaluate Detection Performance

Let's analyze how well RankOD detected the held-out class as outliers.

### Score Statistics

Compare the distribution of scores between normal samples (digits 0-8) and outliers (digit 9).

In [ ]:
print("="*70)
print("Outlier Score Statistics")
print("="*70)

print(f"\nNormal samples (classes 0-8):")
print(f"  Mean:   {np.mean(normal_scores):.4f}")
print(f"  Median: {np.median(normal_scores):.4f}")
print(f"  Std:    {np.std(normal_scores):.4f}")

print(f"\nOutlier samples (class {HELD_OUT_CLASS}):")
print(f"  Mean:   {np.mean(outlier_scores):.4f}")
print(f"  Median: {np.median(outlier_scores):.4f}")
print(f"  Std:    {np.std(outlier_scores):.4f}")

# Calculate separation
separation = (np.mean(outlier_scores) - np.mean(normal_scores)) / np.std(normal_scores)
print(f"\nSeparation: {separation:.2f} standard deviations")
print(f"Score ratio: {np.mean(outlier_scores)/np.mean(normal_scores):.2f}x higher")

### Top-K Outlier Analysis

What percentage of the top-K highest-scoring samples are from the held-out class?

**Perfect detection** would show 100% in all top-K groups.

In [ ]:
print("\nTop-K Outlier Analysis:")
print(f"{'K':>6s} | {'% from class ' + str(HELD_OUT_CLASS):>20s} | {'Count':>6s}")
print(f"{'-'*6}+{'-'*22}+{'-'*6}")

top_k_values = [10, 50, 100, 500, 1000]
for k in top_k_values:
    if k > len(scores):
        continue
    top_k_indices = np.argsort(scores)[-k:]
    top_k_labels = y[top_k_indices]
    pct_outlier = (top_k_labels == HELD_OUT_CLASS).sum() / k * 100
    count_outlier = (top_k_labels == HELD_OUT_CLASS).sum()
    print(f"{k:6d} | {pct_outlier:19.1f}% | {count_outlier:6d}")

### Percentile Analysis

At what percentile do the outliers typically fall? (100th = highest scores)

In [ ]:
# Calculate percentiles for each outlier
percentiles = []
for score in outlier_scores:
    percentile = (scores < score).sum() / len(scores) * 100
    percentiles.append(percentile)

avg_percentile = np.mean(percentiles)
print(f"Average percentile of outliers: {avg_percentile:.1f}th")
print(f"(100th percentile = highest outlier scores)")

## Visualize Results

Let's create comprehensive visualizations to show how well RankOD separates the held-out class.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Score distribution histogram
ax = axes[0, 0]
bins = 50
ax.hist(normal_scores, bins=bins, alpha=0.6, label=f'Normal (classes 0-8)', 
        color='blue', density=True)
ax.hist(outlier_scores, bins=bins, alpha=0.6, label=f'Outlier (class {HELD_OUT_CLASS})', 
        color='red', density=True)
ax.set_xlabel('Outlier Score', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Outlier Scores', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Score by class
ax = axes[0, 1]
classes = sorted(set(y))
class_scores = [scores[y == c] for c in classes]
colors = ['red' if c == HELD_OUT_CLASS else 'blue' for c in classes]
bp = ax.boxplot(class_scores, patch_artist=True)
ax.set_xticklabels(classes)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_xlabel('Digit Class', fontsize=11)
ax.set_ylabel('Outlier Score', fontsize=11)
ax.set_title('Outlier Scores by Digit Class', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. Cumulative distribution
ax = axes[1, 0]
normal_sorted = np.sort(normal_scores)
outlier_sorted = np.sort(outlier_scores)
ax.plot(normal_sorted, np.linspace(0, 100, len(normal_sorted)), 
        label=f'Normal (classes 0-8)', linewidth=2)
ax.plot(outlier_sorted, np.linspace(0, 100, len(outlier_sorted)), 
        label=f'Outlier (class {HELD_OUT_CLASS})', linewidth=2, color='red')
ax.set_xlabel('Outlier Score', fontsize=11)
ax.set_ylabel('Cumulative Percentile', fontsize=11)
ax.set_title('Cumulative Distribution of Outlier Scores', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 4. Top outliers breakdown
ax = axes[1, 1]
top_k_values = [10, 50, 100, 500, 1000]
percentages = []
for k in top_k_values:
    if k > len(scores):
        break
    top_k_indices = np.argsort(scores)[-k:]
    top_k_labels = y[top_k_indices]
    pct = (top_k_labels == HELD_OUT_CLASS).sum() / k * 100
    percentages.append(pct)

bars = ax.bar(range(len(percentages)), percentages, color='red', alpha=0.6)
ax.set_xticks(range(len(percentages)))
ax.set_xticklabels([f'Top {k}' for k in top_k_values[:len(percentages)]])
ax.set_ylabel(f'% from Class {HELD_OUT_CLASS}', fontsize=11)
ax.set_title(f'Percentage of Held-Out Class in Top-K Outliers', fontsize=12, fontweight='bold')
ax.axhline(y=100, color='green', linestyle='--', alpha=0.5, label='Perfect (100%)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 105])

# Add percentage labels on bars
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{pct:.0f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Summary and Conclusions

### Key Findings

In [ ]:
pct_in_top_100 = (y[np.argsort(scores)[-100:]] == HELD_OUT_CLASS).sum() / 100 * 100
pct_in_top_500 = (y[np.argsort(scores)[-500:]] == HELD_OUT_CLASS).sum() / 500 * 100

print("="*70)
print("SUMMARY")
print("="*70)
print(f"""
✓ RankOD successfully identified the held-out class (digit {HELD_OUT_CLASS}) as outliers!

Key findings:
• Outlier samples have {np.mean(outlier_scores)/np.mean(normal_scores):.2f}x higher scores on average
• {pct_in_top_100:.0f}% of top-100 outliers are from the held-out class
• {pct_in_top_500:.0f}% of top-500 outliers are from the held-out class
• The held-out class is separated by {separation:.2f} standard deviations

This demonstrates that RankOD can effectively detect out-of-distribution samples
without seeing them during training, making it useful for anomaly detection in
real-world scenarios where you don't have labeled anomalies.
""")
print("="*70)

### Why This Works

RankOD detects outliers by computing **reverse k-nearest neighbor ranks**:

1. For each point, it finds its k nearest neighbors
2. For each neighbor, it asks: "Where does this point rank in my neighbor's neighborhood?"
3. Normal points appear highly ranked in their neighbors' lists (low rank = high density)
4. Outliers don't appear in their neighbors' lists, or appear very far down (high rank = low density)

Since digit 9 was never seen during training, it forms a distinct cluster that doesn't fit the learned distribution, resulting in low density estimates and high outlier scores.

### Try It Yourself!

Experiment with different configurations:
- Change `HELD_OUT_CLASS` to hold out a different digit (0-9)
- Adjust `N_NEIGHBORS` and `MAX_RANK` to see how they affect detection
- Try without PCA (`X_train = X_normal`, `X_all_pca = X`) for comparison
- Use `detector.predict()` to get binary labels (outlier vs normal)
- Test with different contamination values in `predict(contamination=0.2)`